# AdaptIQ-R: Adaptive Route Optimization Under Disruption
### Computational Intelligence Research Prototype
**Evolutionary Optimization + Pure Python Mamdani Fuzzy Adaptation + Dynamic Disruption Recovery**

This standalone Jupyter Notebook contains the **complete mathematical formulation and execution pipeline** of AdaptIQ-R:
1. **Planar Network Generator**: Euclidean synthetic road network with Kruskal MST (guaranteed connectivity) + $k$-NN edges.
2. **Multi-Objective Dijkstra Fitness**: Distance, traffic delays, customer priorities, and capacity constraints.
3. **Pure Python Mamdani Fuzzy Inference Engine**: Triangular & trapezoidal membership functions, 9 fuzzy rules, min implication, max aggregation, and centroid defuzzification.
4. **Closed-Loop Adaptive GA**: Order Crossover (OX), tournament selection with dynamic pressure, and scale-directed swap mutation.
5. **Real Disruption Injection & Recovery**: Road severance, dynamic fuzzy rule activation, and detour re-routing.
6. **Statistical Comparison**: Baseline GA vs AdaptIQ-R on identical topology.

In [ ]:
# Install dependencies (if running in Google Colab)
# !pip install networkx matplotlib numpy pandas

import math
import random
import time
from typing import List, Dict, Tuple, Optional
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

print("Libraries successfully imported!")

## 1. Synthetic Network Generator
Generates a connected planar-like road network on a $100 \times 100$ coordinate grid.
* Depot is anchored at $(50, 50)$ with node ID `0`.
* Kruskal's Minimum Spanning Tree guarantees overall connectivity.
* $k$-Nearest Neighbors add dense alternate routing paths.

In [ ]:
class Node:
    def __init__(self, node_id: int, x: float, y: float, demand: float = 1.0, priority: int = 1):
        self.id = node_id
        self.x = x
        self.y = y
        self.demand = demand
        self.priority = priority

class Edge:
    def __init__(self, u: int, v: int, distance: float, base_time: float):
        self.u = u
        self.v = v
        self.distance = distance
        self.base_time = base_time
        self.traffic_factor = 1.0
        self.blocked = False

class RoadNetwork:
    def __init__(self, node_count: int = 20, seed: int = 42):
        self.node_count = node_count
        self.seed = seed
        self.depot_id = 0
        self.nodes: Dict[int, Node] = {}
        self.edges: List[Edge] = []
        self.graph = nx.Graph()
        self._generate()

    def _generate(self):
        rng = random.Random(self.seed)
        # 1. Depot at center
        self.nodes[0] = Node(0, 50.0, 50.0, demand=0.0, priority=0)
        
        # 2. Customer nodes
        for i in range(1, self.node_count):
            x = round(rng.uniform(5.0, 95.0), 2)
            y = round(rng.uniform(5.0, 95.0), 2)
            p = rng.choices([1, 2, 3], weights=[0.6, 0.3, 0.1])[0]
            self.nodes[i] = Node(i, x, y, demand=1.0, priority=p)

        # 3. All pairwise distances
        all_pairs = []
        for i in range(self.node_count):
            for j in range(i + 1, self.node_count):
                d = math.hypot(self.nodes[i].x - self.nodes[j].x, self.nodes[i].y - self.nodes[j].y)
                all_pairs.append((d, i, j))
        all_pairs.sort()

        # 4. Kruskal's MST for guaranteed connectivity
        parent = list(range(self.node_count))
        def find(i): 
            if parent[i] == i: return i
            parent[i] = find(parent[i]); return parent[i]
        def union(i, j):
            root_i, root_j = find(i), find(j)
            if root_i != root_j: parent[root_i] = root_j; return True
            return False

        edge_set = set()
        for d, u, v in all_pairs:
            if union(u, v):
                edge_set.add((min(u, v), max(u, v)))

        # 5. k-Nearest Neighbors (k=4) for realistic alternate paths
        for i in range(self.node_count):
            dists = [(math.hypot(self.nodes[i].x - self.nodes[j].x, self.nodes[i].y - self.nodes[j].y), j)
                     for j in range(self.node_count) if i != j]
            dists.sort()
            for d, j in dists[:4]:
                edge_set.add((min(i, j), max(i, j)))

        for u, v in sorted(edge_set):
            dist = round(math.hypot(self.nodes[u].x - self.nodes[v].x, self.nodes[u].y - self.nodes[v].y), 2)
            base_time = round((dist / 40.0) * 60.0, 2) # 40 km/h avg speed
            self.edges.append(Edge(u, v, dist, base_time))
        
        self.rebuild_graph()

    def rebuild_graph(self):
        self.graph.clear()
        for nid, node in self.nodes.items():
            self.graph.add_node(nid, x=node.x, y=node.y, priority=node.priority)
        for e in self.edges:
            if not e.blocked:
                eff_time = e.base_time * e.traffic_factor
                self.graph.add_edge(e.u, e.v, distance=e.distance, effective_time=eff_time)

network = RoadNetwork(node_count=20, seed=42)
print(f"Network built: {len(network.nodes)} nodes, {len(network.edges)} edges. Connected: {nx.is_connected(network.graph)}")

## 2. Multi-Objective Fitness Evaluation
Evaluates a customer permutation $[c_1, c_2, \dots, c_{n-1}]$ anchored at depot $0$.
Uses Dijkstra shortest-path navigation between consecutive stops. If an edge is blocked, Dijkstra automatically routes around it.

In [ ]:
class FitnessEngine:
    def __init__(self, network: RoadNetwork):
        self.network = network
        self.cache = {}

    def get_shortest_path(self, u: int, v: int) -> Tuple[float, float, List[int]]:
        key = (u, v)
        if key in self.cache:
            return self.cache[key]
        try:
            path = nx.shortest_path(self.network.graph, u, v, weight="effective_time")
            dist = sum(self.network.graph[path[i]][path[i+1]]["distance"] for i in range(len(path)-1))
            tt = sum(self.network.graph[path[i]][path[i+1]]["effective_time"] for i in range(len(path)-1))
            res = (dist, tt, path)
        except nx.NetworkXNoPath:
            res = (1e6, 1e6, [])
        self.cache[key] = res
        return res

    def evaluate(self, chromosome: List[int]) -> Dict:
        full_route = [0] + chromosome + [0]
        total_dist = 0.0
        total_time = 0.0
        is_feasible = True
        detailed_path = [0]

        for i in range(len(full_route) - 1):
            u, v = full_route[i], full_route[i+1]
            d, t, p = self.get_shortest_path(u, v)
            if not p:
                is_feasible = False
                total_dist += 1e5
                total_time += 1e5
            else:
                total_dist += d
                total_time += t
                detailed_path.extend(p[1:])

        # Priority Penalty: high priority customers visited late
        priority_pen = 0.0
        for idx, cid in enumerate(chromosome):
            p = self.network.nodes[cid].priority
            if p > 1:
                priority_pen += (idx / len(chromosome)) * (p * 50.0)

        # Weighted normalized scalar fitness (lower is better)
        fitness = (total_dist * 0.001) + (total_time * 0.0005) + (priority_pen * 0.001)
        if not is_feasible:
            fitness += 100.0

        return {
            "fitness": fitness,
            "distance": total_dist,
            "time": total_time,
            "is_feasible": is_feasible,
            "route": full_route,
            "detailed_path": detailed_path
        }

fitness_engine = FitnessEngine(network)
test_chrom = list(range(1, 20))
res = fitness_engine.evaluate(test_chrom)
print(f"Initial chromosome fitness: {res['fitness']:.4f}, distance: {res['distance']:.1f} km")

## 3. Pure Python Mamdani Fuzzy Inference Engine
No external fuzzy libraries. Built directly from mathematical first principles:
* **Triangular Membership**: $\mu(x; a, b, c) = \max(0, \min(\frac{x-a}{b-a}, \frac{c-x}{c-b}))$
* **Trapezoidal Membership**: $\mu(x; a, b, c, d) = \max(0, \min(\frac{x-a}{b-a}, 1, \frac{d-x}{d-b}))$
* **Mamdani Min Implication & Max Aggregation**
* **Centroid Defuzzification** over 100 discretization steps

In [ ]:
def trimf(x: float, a: float, b: float, c: float) -> float:
    if x <= a or x >= c: return 0.0
    if x == b: return 1.0
    if x < b: return (x - a) / (b - a)
    return (c - x) / (c - b)

def trapmf(x: float, a: float, b: float, c: float, d: float) -> float:
    if x <= a or x >= d: return 0.0
    if b <= x <= c: return 1.0
    if x < b: return (x - a) / (b - a)
    return (d - x) / (d - c)

class FuzzyController:
    """Mamdani Fuzzy Controller for online adaptive hyperparameter tuning."""
    def __init__(self):
        pass

    def fuzzify(self, diversity: float, improvement: float, severity: float) -> Dict[str, Dict[str, float]]:
        return {
            "diversity": {
                "LOW": trapmf(diversity, 0.0, 0.0, 0.25, 0.45),
                "MED": trimf(diversity, 0.30, 0.50, 0.70),
                "HIGH": trapmf(diversity, 0.55, 0.75, 1.0, 1.0)
            },
            "improvement": {
                "LOW": trapmf(improvement, 0.0, 0.0, 0.05, 0.15),
                "MED": trimf(improvement, 0.08, 0.20, 0.35),
                "HIGH": trapmf(improvement, 0.25, 0.45, 1.0, 1.0)
            },
            "disruption": {
                "LOW": trapmf(severity, 0.0, 0.0, 0.15, 0.35),
                "MED": trimf(severity, 0.20, 0.45, 0.70),
                "HIGH": trapmf(severity, 0.60, 0.80, 1.0, 1.0)
            }
        }

    def evaluate(self, diversity: float, improvement: float, severity: float) -> Tuple[float, float, List[str]]:
        f = self.fuzzify(diversity, improvement, severity)
        rules = [
            ("R1", min(f["diversity"]["LOW"], f["improvement"]["LOW"]), "HIGH", "HIGH"),
            ("R2", min(f["diversity"]["LOW"], f["improvement"]["MED"]), "HIGH", "MED"),
            ("R3", min(f["diversity"]["HIGH"], f["improvement"]["HIGH"]), "LOW", "LOW"),
            ("R4", f["disruption"]["HIGH"], "HIGH", "HIGH"),
            ("R5", min(f["disruption"]["MED"], f["improvement"]["LOW"]), "HIGH", "MED"),
            ("R6", min(f["disruption"]["LOW"], f["improvement"]["HIGH"]), "LOW", "LOW"),
            ("R7", min(f["diversity"]["MED"], f["improvement"]["MED"]), "MED", "MED"),
            ("R8", min(f["diversity"]["HIGH"], f["improvement"]["LOW"]), "MED", "LOW"),
            ("R9", min(f["disruption"]["LOW"], f["diversity"]["LOW"]), "MED", "MED")
        ]

        # Consequent universes
        mut_universe = np.linspace(0.05, 0.45, 100)
        exp_universe = np.linspace(0.10, 0.90, 100)
        
        mut_mfs = {
            "LOW": [trapmf(x, 0.05, 0.05, 0.12, 0.22) for x in mut_universe],
            "MED": [trimf(x, 0.15, 0.25, 0.35) for x in mut_universe],
            "HIGH": [trapmf(x, 0.28, 0.38, 0.45, 0.45) for x in mut_universe]
        }
        exp_mfs = {
            "LOW": [trapmf(x, 0.10, 0.10, 0.25, 0.45) for x in exp_universe],
            "MED": [trimf(x, 0.30, 0.50, 0.70) for x in exp_universe],
            "HIGH": [trapmf(x, 0.55, 0.75, 0.90, 0.90) for x in exp_universe]
        }

        # Max Aggregation
        mut_agg = np.zeros(100)
        exp_agg = np.zeros(100)
        active_rules = []

        for rid, strength, m_term, e_term in rules:
            if strength > 0.005:
                active_rules.append(rid)
                # Mamdani min implication
                mut_imp = np.minimum(strength, mut_mfs[m_term])
                exp_imp = np.minimum(strength, exp_mfs[e_term])
                mut_agg = np.maximum(mut_agg, mut_imp)
                exp_agg = np.maximum(exp_agg, exp_imp)

        # Centroid Defuzzification
        def defuzz(universe, agg, default):
            denom = np.sum(agg)
            return np.sum(universe * agg) / denom if denom > 1e-9 else default

        mutation_rate = float(defuzz(mut_universe, mut_agg, 0.15))
        exploration_level = float(defuzz(exp_universe, exp_agg, 0.50))
        return mutation_rate, exploration_level, active_rules

fc = FuzzyController()
m_rate, e_level, rules = fc.evaluate(diversity=0.2, improvement=0.01, severity=0.8)
print(f"Fuzzy response to high disruption: Mutation={m_rate:.4f}, Exploration={e_level:.4f}, Rules Fired={rules}")

## 4. Genetic Algorithm with Dynamic Adaptation
* **Order Crossover (OX)**: Generates valid permutations without duplicating customer visits.
* **Scale-Directed Swap Mutation**: Swaps scale with fuzzy exploration output.
* **Adaptive Tournament Selection**: Selection pressure balances exploration vs convergence.

In [ ]:
def order_crossover(parent1: List[int], parent2: List[int], rng: random.Random) -> List[int]:
    size = len(parent1)
    cx1, cx2 = sorted(rng.sample(range(size), 2))
    child = [-1] * size
    child[cx1:cx2+1] = parent1[cx1:cx2+1]
    copied = set(child[cx1:cx2+1])
    fill_vals = [gene for gene in parent2 if gene not in copied]
    ptr = 0
    for i in range(size):
        if child[i] == -1:
            child[i] = fill_vals[ptr]
            ptr += 1
    return child

def adaptive_swap_mutation(chromosome: List[int], mutation_rate: float, exploration_level: float, rng: random.Random) -> List[int]:
    if rng.random() > mutation_rate:
        return list(chromosome)
    mutant = list(chromosome)
    k_swaps = max(1, int(1 + 3 * exploration_level))
    for _ in range(k_swaps):
        i, j = rng.sample(range(len(mutant)), 2)
        mutant[i], mutant[j] = mutant[j], mutant[i]
    return mutant

def calculate_diversity(population: List[List[int]]) -> float:
    # Normalized Hamming / position distance sample
    sample_size = min(20, len(population))
    sample = population[:sample_size]
    n = len(sample[0])
    diffs = 0
    pairs = 0
    for i in range(sample_size):
        for j in range(i + 1, sample_size):
            diffs += sum(1 for k in range(n) if sample[i][k] != sample[j][k])
            pairs += 1
    return (diffs / (pairs * n)) if pairs > 0 else 0.0

class GeneticAlgorithm:
    def __init__(self, network: RoadNetwork, is_adaptive: bool = True, pop_size: int = 60, seed: int = 42):
        self.network = network
        self.is_adaptive = is_adaptive
        self.pop_size = pop_size
        self.rng = random.Random(seed)
        self.fitness_engine = FitnessEngine(network)
        self.fuzzy = FuzzyController()
        self.customer_ids = [nid for nid in network.nodes.keys() if nid != 0]
        self.population = [self.rng.sample(self.customer_ids, len(self.customer_ids)) for _ in range(pop_size)]
        
        self.best_eval = None
        self.best_fitness = float("inf")
        self.history = []
        self.mutation_rate = 0.15
        self.exploration_level = 0.50
        self.disruption_severity = 0.0

    def notify_disruption(self, severity: float):
        self.disruption_severity = severity
        self.fitness_engine.cache.clear()
        # Re-evaluate population after disruption
        evals = [self.fitness_engine.evaluate(ind) for ind in self.population]
        best_idx = min(range(len(evals)), key=lambda i: evals[i]["fitness"])
        self.best_eval = evals[best_idx]
        self.best_fitness = self.best_eval["fitness"]

    def step(self, gen_idx: int) -> Dict:
        # 1. Evaluate fitness
        evals = [self.fitness_engine.evaluate(ind) for ind in self.population]
        fitnesses = [e["fitness"] for e in evals]
        
        best_idx = min(range(len(fitnesses)), key=lambda i: fitnesses[i])
        curr_best_eval = evals[best_idx]
        curr_best_fit = curr_best_eval["fitness"]
        
        # Improvement metric
        prev_fit = self.best_fitness if self.best_fitness != float("inf") else curr_best_fit
        impr = max(0.0, (prev_fit - curr_best_fit) / prev_fit) if prev_fit > 0 else 0.0
        if curr_best_fit < self.best_fitness:
            self.best_fitness = curr_best_fit
            self.best_eval = curr_best_eval

        # 2. Diversity
        diversity = calculate_diversity(self.population)

        # 3. Fuzzy Adaptation (if enabled)
        rules_fired = []
        if self.is_adaptive:
            self.mutation_rate, self.exploration_level, rules_fired = self.fuzzy.evaluate(
                diversity=diversity, improvement=impr, severity=self.disruption_severity
            )
        else:
            self.mutation_rate = 0.15
            self.exploration_level = 0.50

        # 4. Reproduction with Elitism
        new_pop = [list(curr_best_eval["route"][1:-1])]
        k_tourn = max(2, int(2 + 3 * (1.0 - self.exploration_level)))
        
        while len(new_pop) < self.pop_size:
            # Tournament
            cand1 = self.rng.sample(range(self.pop_size), k_tourn)
            p1 = self.population[min(cand1, key=lambda i: fitnesses[i])]
            cand2 = self.rng.sample(range(self.pop_size), k_tourn)
            p2 = self.population[min(cand2, key=lambda i: fitnesses[i])]
            
            child = order_crossover(p1, p2, self.rng)
            child = adaptive_swap_mutation(child, self.mutation_rate, self.exploration_level, self.rng)
            new_pop.append(child)

        self.population = new_pop
        
        stat = {
            "gen": gen_idx,
            "best_fitness": self.best_fitness,
            "avg_fitness": np.mean(fitnesses),
            "diversity": diversity,
            "mutation": self.mutation_rate,
            "exploration": self.exploration_level,
            "rules": rules_fired
        }
        self.history.append(stat)
        return stat

## 5. Execution Pipeline: Baseline vs AdaptIQ-R Under Disruption
We run the complete experimental workflow:
1. **Phase 1 (Gens 1-40)**: Optimize on clean road network.
2. **Phase 2 (Gen 40)**: Inject a severe road block disruption (sever edge $8 \leftrightarrow 11$).
3. **Phase 3 (Gens 41-80)**: Adaptive recovery phase.

In [ ]:
# Run Baseline GA
net_base = RoadNetwork(node_count=20, seed=42)
ga_base = GeneticAlgorithm(net_base, is_adaptive=False, pop_size=60, seed=42)
for g in range(1, 41):
    ga_base.step(g)

# Disrupt Baseline
net_base.edges[12].blocked = True
net_base.rebuild_graph()
ga_base.notify_disruption(severity=0.5)
for g in range(41, 81):
    ga_base.step(g)

# Run AdaptIQ-R (Fuzzy Adaptive)
net_adapt = RoadNetwork(node_count=20, seed=42)
ga_adapt = GeneticAlgorithm(net_adapt, is_adaptive=True, pop_size=60, seed=42)
for g in range(1, 41):
    ga_adapt.step(g)

# Disrupt AdaptIQ-R
net_adapt.edges[12].blocked = True
net_adapt.rebuild_graph()
ga_adapt.notify_disruption(severity=0.5)
for g in range(41, 81):
    ga_adapt.step(g)

print(f"Baseline GA Recovered Fitness: {ga_base.best_fitness:.4f}")
print(f"AdaptIQ-R Recovered Fitness:   {ga_adapt.best_fitness:.4f}")
adv = ((ga_base.best_fitness - ga_adapt.best_fitness) / ga_base.best_fitness) * 100
print(f"AdaptIQ-R Advantage:           {adv:+.2f}%")

## 6. Visualization & Convergence Curves
Plotting convergence trajectories, parameter adaptation, and the physical route on the network.

In [ ]:
plt.figure(figsize=(15, 5))

# 1. Fitness Convergence
plt.subplot(1, 3, 1)
b_fit = [h["best_fitness"] for h in ga_base.history]
a_fit = [h["best_fitness"] for h in ga_adapt.history]
plt.plot(range(1, 81), b_fit, label="Baseline GA (Static 0.15)", color="gray", linestyle="--")
plt.plot(range(1, 81), a_fit, label="AdaptIQ-R (Fuzzy Adaptive)", color="cyan", linewidth=2)
plt.axvline(x=40, color="red", linestyle=":", label="Disruption Injected")
plt.title("Fitness Trajectory Under Disruption")
plt.xlabel("Generation")
plt.ylabel("Best Fitness (Cost)")
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Fuzzy Parameter Adaptation
plt.subplot(1, 3, 2)
mut_history = [h["mutation"] for h in ga_adapt.history]
exp_history = [h["exploration"] for h in ga_adapt.history]
div_history = [h["diversity"] for h in ga_adapt.history]
plt.plot(range(1, 81), mut_history, label="Mutation Rate", color="orange", linewidth=2)
plt.plot(range(1, 81), exp_history, label="Exploration Level", color="purple", linewidth=2)
plt.plot(range(1, 81), div_history, label="Population Diversity", color="green", linestyle="--")
plt.axvline(x=40, color="red", linestyle=":")
plt.title("AdaptIQ-R Online Fuzzy Telemetry")
plt.xlabel("Generation")
plt.ylabel("Parameter Value [0, 1]")
plt.legend()
plt.grid(True, alpha=0.3)

# 3. Route Map
plt.subplot(1, 3, 3)
coords = {n.id: (n.x, n.y) for n in net_adapt.nodes.values()}
for e in net_adapt.edges:
    if e.blocked:
        plt.plot([coords[e.u][0], coords[e.v][0]], [coords[e.u][1], coords[e.v][1]], 'r--', linewidth=3, label="Blocked Road" if e == net_adapt.edges[12] else "")
    else:
        plt.plot([coords[e.u][0], coords[e.v][0]], [coords[e.u][1], coords[e.v][1]], color='#cccccc', linewidth=1)

# Plot recovered route
rec_route = ga_adapt.best_eval["detailed_path"]
rx = [coords[nid][0] for nid in rec_route]
ry = [coords[nid][1] for nid in rec_route]
plt.plot(rx, ry, color="#10b981", linewidth=2, label="Recovered Detour Route")

# Plot Nodes
for nid, node in net_adapt.nodes.items():
    if nid == 0:
        plt.scatter(node.x, node.y, color="gold", s=120, marker="*", zorder=5, label="Depot (0)")
    else:
        plt.scatter(node.x, node.y, color="#06b6d4", s=40, zorder=4)

plt.title("Recovered Physical Detour Route")
plt.legend(loc="upper right", fontsize=8)
plt.axis("off")

plt.tight_layout()
plt.show()